# 01 — Search and Download NISAR Data

Discover NISAR products using both **ASF** (`asf_search`) and **NASA Earthdata**
(`earthaccess`), inspect search results, and optionally download granules to a
local directory.

> **Tip:** Cloud-native access (S3 or HTTPS streaming) is preferred over bulk
> downloads. See notebook 02 for direct read.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/nice-sar/blob/main/notebooks/01_search_and_download.ipynb)

## 1. Install and Import

In [ ]:
%pip install -q --upgrade --force-reinstall --no-cache-dir \
    "fsspec<=2025.3.0" \
    "s3fs<=2025.3.0" \
    "git+https://github.com/bullocke/nice-sar.git@main"

In [ ]:
import logging

from nice_sar.auth.earthdata import login
from nice_sar.search.asf import get_result_size_bytes, search_gcov, search_nisar
from nice_sar.search.earthdata import search_earthdata

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s", force=True)
logger = logging.getLogger(__name__)

## 2. Authenticate with NASA Earthdata

Required for both ASF and Earthdata searches.

In [ ]:
login()

## 3. Search via ASF (`asf_search`)

`search_gcov()` is a convenience wrapper around `search_nisar(product_type="GCOV")`.
Both use `dataset='NISAR'` under the hood.

In [ ]:
# Search for NISAR GCOV products over Salt Lake City, UT
bbox = (-112.1, 40.5, -111.7, 40.9)
asf_results = search_gcov(bbox=bbox, max_results=10)

logger.info("ASF results: %d granules", len(asf_results))
for r in asf_results[:3]:
    logger.info("  %s", r.properties.get("fileName", r))

## 4. Search via Earthdata (`earthaccess`)

`search_earthdata()` uses the CMR API with collection short names.
The default short name for NISAR GCOV is `NISAR_L2_GCOV_BETA_V1`.

In [ ]:
ed_results = search_earthdata(bbox=bbox, count=10)

logger.info("Earthdata results: %d granules", len(ed_results))
for g in ed_results[:3]:
    logger.info("  %s", g["meta"]["concept-id"])

## 5. Inspect Search Result Properties

ASF results carry metadata like file size, processing level, and download URLs.

In [ ]:
if asf_results:
    r = asf_results[0]
    props = r.properties
    size_bytes = get_result_size_bytes(r)

    logger.info("Granule: %s", props.get("fileName", "N/A"))
    logger.info("  Processing Level: %s", props.get("processingLevel", "N/A"))
    logger.info("  Start Time: %s", props.get("startTime", "N/A"))
    logger.info("  Frame: %s", props.get("frameNumber", "N/A"))
    logger.info("  URL: %s", props.get("url", "N/A"))
    if size_bytes is not None:
        logger.info("  Size: %.2f MB", size_bytes / 1e6)
    else:
        logger.info(
            "  Size metadata: %s",
            props.get("bytes") if props.get("bytes") is not None else "N/A",
        )
else:
    logger.warning("No ASF results found — try a different bbox or date range.")

## 6. Search Other Product Types

`search_nisar()` supports any NISAR product type: RSLC, GSLC, GCOV, GUNW, GOFF.

In [ ]:
for product in ["RSLC", "GSLC", "GCOV", "GUNW"]:
    results = search_nisar(product_type=product, bbox=bbox, max_results=5)
    logger.info("  %s: %d results", product, len(results))

## 7. Download Granules (Optional)

Use `download_url()` or `download_granules()` if you need local files.
**Cloud-native streaming is preferred** — see notebooks 02 and 04.

> ⚠️ Downloads can be **very large** (multi-GB per granule). Only download
> when S3/HTTPS streaming isn't sufficient.

In [ ]:
# Example (commented out to avoid accidental large downloads):
#
# from nice_sar.io.download import download_url
# from pathlib import Path
#
# output_dir = Path("./downloads")
# url = asf_results[0].properties["url"]
# local_file = download_url(url, output_dir)
# logger.info("Downloaded: %s", local_file)

logger.info("Skipping download — use streaming access in notebooks 02/04.")